## Lab Week 9 - Lotka-Volterra Model
---
### Simulasi dan Pemodelan Predator-Prey

- Nama: Faaid Sakhaa
- NIM: 24/539398/TK/59820
- Kelas: Teknologi Informasi

Notebook ini mengerjakan deliverable:
1. Time-series prey dan predator
2. Phase portrait dengan titik ekuilibrium ditandai
3. Parameter sweep (3 set) dan perbandingan amplitudo osilasi

Model dasar:
$$\frac{dx}{dt} = \alpha x - \beta x y$$
$$\frac{dy}{dt} = \delta x y - \gamma y$$

Penjelasan model dasar:
| Simbol | Makna |
|---|---|
| $x(t)$ | Populasi prey (mangsa) pada waktu $t$ |
| $y(t)$ | Populasi predator pada waktu $t$ |
| $\alpha$ | Laju pertumbuhan alami prey |
| $\beta$ | Intensitas interaksi predasi (pertemuan prey-predator) |
| $\delta$ | Efisiensi konversi hasil predasi menjadi pertumbuhan predator |
| $\gamma$ | Laju kematian alami predator |

Interpretasi tiap suku:
- $+\alpha x$: prey bertambah karena reproduksi alami.
- $-\beta xy$: prey berkurang karena dimangsa predator.
- $+\delta xy$: predator bertambah karena ketersediaan makanan (prey).
- $-\gamma y$: predator berkurang karena kematian alami.


### 1. Import Library dan Definisi Fungsi
Mengimpor `numpy` dan `matplotlib`, lalu membuat fungsi model Lotka-Volterra serta integrator RK4 untuk simulasi numerik.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def lotka_volterra_rhs(t, state, alpha, beta, delta, gamma, H=0.0):
    """Lotka-Volterra RHS with optional prey harvesting term -H."""
    x, y = state
    dx = alpha * x - beta * x * y - H
    dy = delta * x * y - gamma * y
    return np.array([dx, dy], dtype=float)

def rk4_integrate(rhs, y0, t, **params):
    y = np.zeros((len(t), len(y0)), dtype=float)
    y[0] = y0
    dt = t[1] - t[0]

    for i in range(len(t) - 1):
        yi = y[i]
        ti = t[i]
        k1 = rhs(ti, yi, **params)
        k2 = rhs(ti + 0.5 * dt, yi + 0.5 * dt * k1, **params)
        k3 = rhs(ti + 0.5 * dt, yi + 0.5 * dt * k2, **params)
        k4 = rhs(ti + dt, yi + dt * k3, **params)
        y[i + 1] = yi + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        y[i + 1] = np.maximum(y[i + 1], 0.0)
    return y

def coexistence_equilibrium(alpha, beta, delta, gamma, H=0.0):
    x_star = gamma / delta
    y_star = (alpha * x_star - H) / (beta * x_star)
    return x_star, y_star


### 2. Setup Parameter dan Simulasi Dasar
Mendefinisikan 3 set parameter, memilih kondisi awal yang sama dan tidak tepat berada pada titik ekuilibrium, lalu menjalankan simulasi untuk tiap set.


In [2]:
# Suggested parameter sets from the assignment slide
parameter_sets = [
    {'name': 'Set 1', 'alpha': 1.0, 'beta': 0.10, 'delta': 0.075, 'gamma': 1.5},
    {'name': 'Set 2', 'alpha': 0.5, 'beta': 0.10, 'delta': 0.075, 'gamma': 1.5},
    {'name': 'Set 3', 'alpha': 1.0, 'beta': 0.05, 'delta': 0.075, 'gamma': 1.5},
]

# The shared initial condition is chosen away from the equilibrium points
# so each parameter set displays its own transient and oscillatory behavior.
x0, y0 = 30.0, 8.0
t_end, dt = 200.0, 0.05
t = np.arange(0.0, t_end + dt, dt)

results = {}
equilibrium_rows = []
for p in parameter_sets:
    params = {k: v for k, v in p.items() if k != 'name'}
    sol = rk4_integrate(lotka_volterra_rhs, np.array([x0, y0]), t, **params)
    results[p['name']] = sol
    x_star_i, y_star_i = coexistence_equilibrium(**params)
    equilibrium_rows.append((p['name'], x_star_i, y_star_i))

set1 = parameter_sets[0]
sol1 = results['Set 1']
x1, y1 = sol1[:, 0], sol1[:, 1]

x_star, y_star = coexistence_equilibrium(**{k: v for k, v in set1.items() if k != 'name'})
print('Non-trivial equilibria for all parameter sets:')
for name, x_star_i, y_star_i in equilibrium_rows:
    print(f"{name:5s} | x* = {x_star_i:6.3f} | y* = {y_star_i:6.3f}")


Non-trivial equilibria for all parameter sets:
Set 1 | x* = 20.000 | y* = 10.000
Set 2 | x* = 20.000 | y* =  5.000
Set 3 | x* = 20.000 | y* = 20.000
